# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

**Opdracht:** Bouw een voorspellend model om de uitkomst van voetbalwedstrijden (winst, verlies of gelijkspel) te voorspellen en geef advies aan de technisch directeur.

---
## 1. Data Inladen en Exploratie

### 1.1 Benodigde Imports

In [ ]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import tempfile
import os

# Set up plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'modules'

### 1.2 Database Verbinding

In [ ]:
# Set up Kaggle credentials and download data
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

### 1.3 Beschikbare Tabellen

In [ ]:
# See all table names and their structures
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Beschikbare tabellen:")
for table in tables:
    print(f"  - {table[0]}")

# Get table information
for table in tables:
    table_name = table[0]
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = cursor.fetchall()
    print(f"\n{table_name}:")
    for col in columns:
        print(f"  - {col[1]} ({col[2]})")

### 1.4 Data Laden in DataFrames

In [ ]:
# Load all tables into DataFrames
country_df = pd.read_sql("SELECT * FROM Country", connection)
league_df = pd.read_sql("SELECT * FROM League", connection)
team_df = pd.read_sql("SELECT * FROM Team", connection)
match_df = pd.read_sql("SELECT * FROM Match", connection)
player_df = pd.read_sql("SELECT * FROM Player", connection)
player_attributes_df = pd.read_sql("SELECT * FROM Player_Attributes", connection)

print("DataFrames loaded successfully!")
print(f"\nAantal records per tabel:")
print(f"  Countries: {len(country_df)}")
print(f"  Leagues: {len(league_df)}")
print(f"  Teams: {len(team_df)}")
print(f"  Matches: {len(match_df)}")
print(f"  Players: {len(player_df)}")
print(f"  Player Attributes: {len(player_attributes_df)}")

---
## 2. Data Beschrijving en Relaties

### 2.1 Tabel Overzicht

In [ ]:
# Show sample data from each table
print("COUNTRY TABLE:")
display(country_df.head())

print("\nLEAGUE TABLE:")
display(league_df.head())

print("\nTEAM TABLE:")
display(team_df.head())

print("\nMATCH TABLE:")
display(match_df.head())

print("\nPLAYER TABLE:")
display(player_df.head())

print("\nPLAYER_ATTRIBUTES TABLE:")
display(player_attributes_df.head())

### 2.2 Relaties tussen Tabellen

In [ ]:
# Show relationships
print("TABEL RELATIES:")
print("\n1. Country <-> League")
print(f"   - Country.id (rows: {country_df['id'].nunique()}) <-> League.country_id (rows: {league_df['country_id'].nunique()})")
print(f"   - Relatie: 1 Land kan meerdere competities hebben")

print("\n2. League <-> Match")
print(f"   - League.id (rows: {league_df['id'].nunique()}) <-> Match.league_id (rows: match_df['league_id'].nunique() if 'league_id' in match_df.columns else 'N/A')")
print(f"   - Relatie: 1 Competitie kan meerdere wedstrijden hebben")

print("\n3. Team <-> Match")
print(f"   - Team.team_api_id <-> Match.home_team_api_id / Match.away_team_api_id")
print(f"   - Relatie: 1 Team kan als home of away team in meerdere wedstrijden spelen")

print("\n4. Team <-> Player")
print(f"   - Team.team_api_id <-> Match.home_player_X / Match.away_player_X")
print(f"   - Relatie: 1 Team heeft meerdere spelers")

print("\n5. Player <-> Player_Attributes")
print(f"   - Player.player_api_id (rows: {player_df['player_api_id'].nunique()}) <-> Player_Attributes.player_api_id (rows: {player_attributes_df['player_api_id'].nunique()})")
print(f"   - Relatie: 1 Speler kan meerdere attributen hebben voor verschillende data")

---
## 3. Club Selectie en Identifiers

### 3.1 Beschikbare Clubs en Competities

In [ ]:
# Show available clubs and leagues
print("Beschikbare competities:")
league_info = league_df[['id', 'name', 'country_id']].drop_duplicates()
display(league_info)

print("\nBeschikbare clubs:")
print(f"Totaal: {len(team_df)} clubs")
display(team_df[['team_api_id', 'team_long_name', 'team_short_name']].head(20))

### 3.2 Club Setup

**TODO: Kies hier een club uit de dataset**

Verander CLUB_NAME en CLUB_ID naar je gekozen club.

In [ ]:
# Select your club
# TODO: Change these values to your selected club
CLUB_ID = 9825  # Manchester United
CLUB_NAME = "Manchester United"
SELECTED_SEASON = "2015/2016"  # Change to desired season

# Create Club object
my_club = Club(CLUB_ID, CLUB_NAME)

# Load matches for the club
my_club.load_matches(match_df)

# Get basic info
print(f"Club: {my_club.team_name}")
print(f"Total matches in database: {len(my_club.matches_df)}")
print(f"Seasons available: {my_club.matches_df['season'].unique()}")

---
## 4. Seizoen Analyse en Eindstand

### 4.1 Wedstrijden van het Seizoen

In [ ]:
# Get season matches
season_matches = my_club.get_season_matches(SELECTED_SEASON)

print(f"Wedstrijden {CLUB_NAME} - Seizoen {SELECTED_SEASON}")
print(f"Totaal: {len(season_matches)} wedstrijden")
print(f"\nThuis wedstrijden: {len(season_matches[season_matches['home_team_api_id'] == CLUB_ID])}")
print(f"Uit wedstrijden: {len(season_matches[season_matches['away_team_api_id'] == CLUB_ID])}")

display(season_matches[[col for col in season_matches.columns if col in 
                         ['date', 'home_team_api_id', 'away_team_api_id', 'home_team_goal', 'away_team_goal']]].head(10))

### 4.2 Seizoen Statistieken

In [ ]:
# Calculate season statistics
stats = my_club.calculate_stats(SELECTED_SEASON)

print(f"Statistieken {CLUB_NAME} - Seizoen {SELECTED_SEASON}")
print(f"\nWedstrijden gespeeld: {stats['matches_played']}")
print(f"Wins: {stats['wins']}")
print(f"Draws: {stats['draws']}")
print(f"Losses: {stats['losses']}")
print(f"\nDoelpunten voor: {stats['goals_for']}")
print(f"Doelpunten tegen: {stats['goals_against']}")
print(f"Doelsaldo: {stats['goal_difference']}")
print(f"\nPunten totaal: {stats['points']}")
print(f"Gemiddelde punten per wedstrijd: {stats['points'] / stats['matches_played']:.2f}")

### 4.3 Eindstand Competitie

---
## 5. Feature Engineering voor ML Model

In [ ]:
# Create features for ML model
# Features: Team performance metrics, historical performance, etc.

ml_features = []

for idx, match in season_all_matches.iterrows():
    home_team_id = match['home_team_api_id']
    away_team_id = match['away_team_api_id']
    
    # Get matches BEFORE this match for each team
    prev_home_matches = season_all_matches[
        (season_all_matches['date'] < match['date']) &
        ((season_all_matches['home_team_api_id'] == home_team_id) |
         (season_all_matches['away_team_api_id'] == home_team_id))
    ]
    
    prev_away_matches = season_all_matches[
        (season_all_matches['date'] < match['date']) &
        ((season_all_matches['home_team_api_id'] == away_team_id) |
         (season_all_matches['away_team_api_id'] == away_team_id))
    ]
    
    # Calculate rolling statistics
    if len(prev_home_matches) > 0:
        home_goals_for = (prev_home_matches[prev_home_matches['home_team_api_id'] == home_team_id]['home_team_goal'].sum() +
                          prev_home_matches[prev_home_matches['away_team_api_id'] == home_team_id]['away_team_goal'].sum())
        home_goals_against = (prev_home_matches[prev_home_matches['home_team_api_id'] == home_team_id]['away_team_goal'].sum() +
                             prev_home_matches[prev_home_matches['away_team_api_id'] == home_team_id]['home_team_goal'].sum())
    else:
        home_goals_for = 0
        home_goals_against = 0
    
    if len(prev_away_matches) > 0:
        away_goals_for = (prev_away_matches[prev_away_matches['home_team_api_id'] == away_team_id]['home_team_goal'].sum() +
                         prev_away_matches[prev_away_matches['away_team_api_id'] == away_team_id]['away_team_goal'].sum())
        away_goals_against = (prev_away_matches[prev_away_matches['home_team_api_id'] == away_team_id]['away_team_goal'].sum() +
                             prev_away_matches[prev_away_matches['away_team_api_id'] == away_team_id]['home_team_goal'].sum())
    else:
        away_goals_for = 0
        away_goals_against = 0
    
    # Determine outcome
    if match['home_team_goal'] > match['away_team_goal']:
        outcome = 'H'  # Home win
    elif match['home_team_goal'] < match['away_team_goal']:
        outcome = 'A'  # Away win
    else:
        outcome = 'D'  # Draw
    
    ml_features.append({
        'date': match['date'],
        'home_team_id': home_team_id,
        'away_team_id': away_team_id,
        'home_goals_for': home_goals_for,
        'home_goals_against': home_goals_against,
        'away_goals_for': away_goals_for,
        'away_goals_against': away_goals_against,
        'home_goal_diff': home_goals_for - home_goals_against,
        'away_goal_diff': away_goals_for - away_goals_against,
        'outcome': outcome
    })

ml_df = pd.DataFrame(ml_features)

print(f"ML Features DataFrame created with {len(ml_df)} matches")
display(ml_df.head(10))

---
## 6. Predictive Model Development

**TODO: Build and train ML model here**

In [ ]:
# Example structure for model development
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare features and target
X = ml_df[['home_goals_for', 'home_goals_against', 'away_goals_for', 'away_goals_against', 
            'home_goal_diff', 'away_goal_diff']]
y = ml_df['outcome']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

---
## 7. Conclusies en Advies aan Technisch Directeur

### 7.1 Belangrijkste Bevindingen

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print("Feature Importance:")
display(feature_importance)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Match Outcome Prediction')
plt.tight_layout()
plt.show()

### 7.2 Advies voor Technisch Directeur

**Gebaseerd op de analyse, hier zijn aanbevelingen voor het volgende seizoen:**

1. **Focus op Verdediging**: [Add specific findings]
2. **Aanvallende Kracht**: [Add specific findings]
3. **Speelstijl Optimalisatie**: [Add specific findings]
4. **Speler Selectie**: [Add specific findings]

**Verwachte Impact**: Door deze aanbevelingen toe te passen, kan de club naar verwachting [X] meer punten behalen in het volgende seizoen.